# Milestone 1 — arXiv Corpus Collection and Benchmark

Khánh: discovery/metadata · Khang: sources · Nguyên: references · Danh: pipeline/metrics.

## 1. Setup

Local: dùng kernel `arxiv-project`. Colab: chọn accelerator **None**.
Các chế độ: `cold` tạo lượt mới, `resume` chạy tiếp, `analyze` đọc kết quả đã lưu.


In [ ]:
import sys
print(sys.executable)

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

ALLOW_NETWORK = True
READ_COLAB_SECRET = False
REPOSITORY_URL = 'https://github.com/Hireophant/arXivProject.git'
REPOSITORY_BRANCH = 'master'

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ModuleNotFoundError, ValueError):
    IN_COLAB = False
if IN_COLAB:
    import google.colab

# Enable source setup automatically on Colab; use the local environment otherwise.
CLONE_REPOSITORY = IN_COLAB
INSTALL_DEPENDENCIES = IN_COLAB

PROJECT_DIR = next((parent for parent in [Path.cwd(), *Path.cwd().parents]
                    if (parent / 'src/crawling/__init__.py').is_file()), None)
if PROJECT_DIR is None:
    PROJECT_DIR = Path('/content/arXivProject') if IN_COLAB else Path.cwd() / 'arXivProject'

if CLONE_REPOSITORY:
    if not ALLOW_NETWORK:
        raise ValueError('Enable ALLOW_NETWORK before cloning.')
    if PROJECT_DIR.exists():
        if not (PROJECT_DIR / 'src/crawling/__init__.py').is_file():
            raise ValueError('Clone destination already exists without project source; choose a different directory.')
        print('Using existing checkout:', PROJECT_DIR)
    else:
        subprocess.run(['git', 'clone', '--branch', REPOSITORY_BRANCH, '--single-branch',
                        REPOSITORY_URL, str(PROJECT_DIR)], check=True)

if not (PROJECT_DIR / 'src/crawling/__init__.py').is_file():
    raise FileNotFoundError('Upload the project source or enable ALLOW_NETWORK and CLONE_REPOSITORY, then rerun setup.')
if sys.version_info < (3, 11):
    raise RuntimeError('Python >= 3.11 is required.')
if INSTALL_DEPENDENCIES:
    if not ALLOW_NETWORK:
        raise ValueError('Enable ALLOW_NETWORK before installing packages.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                    str(PROJECT_DIR / 'src/requirements.txt')], check=True)
if READ_COLAB_SECRET:
    if not IN_COLAB:
        raise RuntimeError('Colab Secrets can only be read in Colab.')
    from google.colab import userdata
    os.environ['S2_API_KEY'] = userdata.get('S2_API_KEY')
    print('Semantic Scholar credential loaded; its value is not displayed.')

SRC = PROJECT_DIR / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('Source:', PROJECT_DIR, '| Runtime:', 'Colab' if IN_COLAB else 'local')

Khóa Semantic Scholar tùy chọn: biến môi trường `S2_API_KEY` hoặc Colab Secrets (`READ_COLAB_SECRET`).


In [ ]:
import hashlib
import importlib.metadata
import json
import logging
import platform
import shutil
import time
import zipfile
import uuid
from copy import deepcopy
from datetime import datetime, timezone

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import pandas as pd
import psutil
from IPython.display import display, Markdown, HTML

from crawling.pipeline import run_pipeline, load_manifest, initialize_manifest, save_manifest, paper_directory, _cpu_only_environment, _assignment_scope
from crawling.download import _cached as cached_source_result
from crawling.failures import failure_rows, quarantined_ids
from crawling.presentation import (preview_table, preview_style, compact_text, format_bytes, format_duration,
                                    summary_dashboard, verified_drive_bytes, verified_drive_source_files, export_issue_groups)
from crawling.metrics import summarize_corpus, export_report_data
from crawling.discover import OaiClient, OAI_ENDPOINT, METADATA_PREFIX, METADATA_SOURCE
from crawling.backup import backup_run
from crawling.sessions import prepare_session, seed_pending, finish_session, scope_ids, progress, plan_session, migrate_permanent_failures
from crawling.assignments import GROUP_ASSIGNMENTS, ORIGINAL_GROUP_ASSIGNMENTS, VERIFIED_MONTH_END_IDS
from crawling.checkpoints import BackgroundCheckpoints
from crawling.references import _cached_result as cached_reference_result
from crawling.dates import HYBRID_SOURCE
from crawling.reports import write_session_breakdown

logging.basicConfig(level=logging.INFO, format="%(message)s")
VERBOSE_LOGS = False  # Enable per-ID discovery logs only when debugging.
logging.getLogger("crawling.discover").setLevel(logging.INFO if VERBOSE_LOGS else logging.WARNING)
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_colwidth', 80)

def show_run_dashboard(backup=None, invalid_ids=()):
    drive_size = None
    drive_files = None
    if DRIVE_READY and RUN_MODE != 'analyze':
        try:
            drive_size = verified_drive_bytes(SESSION_STATE_PATH, SELECTED_MEMBER, run_result['run_id'],
                extra_receipt=backup['receipt_path'] if backup else None)
            drive_files = verified_drive_source_files(SESSION_STATE_PATH, SELECTED_MEMBER, run_result['run_id'],
                extra_receipt=backup['receipt_path'] if backup else None)
        except (OSError, ValueError, KeyError, zipfile.BadZipFile) as exc:
            print('Không xác minh được backup Drive:', type(exc).__name__)
    session_seconds = (time.perf_counter() - SESSION_STARTED_AT
                       if RUN_MODE != 'analyze' and SESSION_STARTED_AT is not None else None)
    display(HTML(summary_dashboard(manifest, run_result, drive_bytes=drive_size, invalid_ids=invalid_ids,
                 drive_source_files=drive_files, session_seconds=session_seconds,
                 session_finished=backup is not None and drive_files is not None)))

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 180, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

## 2. Assigned ranges

Chọn `SELECTED_MEMBER` và `NEW_PAPERS_PER_SESSION`. Mỗi dải 5.000 ID được chia thành ba đoạn không giao nhau: 1.667 / 1.667 / 1.666 ID.
Tiến độ từng tài khoản được lưu trên Drive; giữ nguyên bảng phân công và Drive root giữa các lượt.


In [ ]:
ASSIGNMENTS = deepcopy(GROUP_ASSIGNMENTS)
SELECTED_MEMBER = 'Danh'
MONTH_END_IDS = dict(VERIFIED_MONTH_END_IDS)
RUN_MODE = 'cold'
EXISTING_ROOT = None
REQUIRE_COLAB_CPU = IN_COLAB
CREATE_REPORT_ZIP = True
DOWNLOAD_REPORT_ZIP = False  # Drive keeps the report; browser download is optional.
# Drive connection and backup options.
SAVE_TO_DRIVE = IN_COLAB
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_BACKUP_DIR = DRIVE_MOUNT_POINT / 'MyDrive/arXivProject/Milestone1'
AUTO_RETRY_FROM_DRIVE = IN_COLAB
SCOPE_PURPOSE = 'assigned'
# Session size and retry limits.
AUTO_PLAN_SESSIONS = True
NEW_PAPERS_PER_SESSION = 100
MAX_RETRY_PAPERS = 40
RETRY_ONLY = False
RETRY_PERMANENT_FAILURES = False  # Explicit resume only, after reviewing upstream evidence.
BACKGROUND_CHECKPOINTS = True
CHECKPOINT_EVERY_PAPERS = 40

# Cornell metadata: build/download once, then reuse a verified local SQLite copy.
USE_HYBRID_REFERENCE_DATES = True
USE_KAGGLE_REFERENCE_DATES = True
AUTO_DOWNLOAD_KAGGLE = IN_COLAB
KAGGLE_SNAPSHOT_PATH = None  # Optional local JSONL or Kaggle ZIP for the first build.
REFRESH_KAGGLE_INDEX = False  # Explicit rebuild; old immutable indexes/backups survive.
V1_CACHE_MAX_AGE_DAYS = 365
V1_CACHE_DIR = None  # Optional separate EMPTY path for a cold benchmark; keep normal progress untouched.

POLICY = {
    'timeout_seconds': 30.0, 'max_attempts': 3,
    'arxiv_delay_seconds': 3.1, 's2_delay_seconds': 3.1,
    'backoff_cap_seconds': 120.0, 'user_agent': 'ArxivCourseProject/0.2 (HCMUS student project)',
}
BASE_DATA_DIR = Path('/content/arxiv-data') if IN_COLAB else PROJECT_DIR / 'data/milestone1'
RUN_FOLDER = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]

if RETRY_PERMANENT_FAILURES and RUN_MODE != 'resume':
    raise ValueError('Permanent retries require an explicit resume of an existing corpus.')
if RUN_MODE not in {'cold', 'resume', 'analyze'}:
    raise ValueError('RUN_MODE must be cold, resume or analyze.')
if len(ASSIGNMENTS) != 12 or len({row['member'] for row in ASSIGNMENTS}) != 12:
    raise ValueError('Provide twelve assignments with distinct account names.')
selected = [row for row in ASSIGNMENTS if row['member'] == SELECTED_MEMBER]
if len(selected) != 1:
    raise ValueError('SELECTED_MEMBER must match one assignment.')
selected_assignment = selected[0]
member_number = ASSIGNMENTS.index(selected_assignment) + 1
SCOPE = None
ASSIGNMENT_SCOPE = _assignment_scope(selected_assignment, MONTH_END_IDS)
previous_assignment = next((row for row in ORIGINAL_GROUP_ASSIGNMENTS if row['member'] == SELECTED_MEMBER), None)
PREVIOUS_ASSIGNMENT_SCOPE = (_assignment_scope(previous_assignment, MONTH_END_IDS)
                             if previous_assignment is not None else None)

if RUN_MODE == 'analyze':
    if EXISTING_ROOT is None:
        raise ValueError('Set EXISTING_ROOT to analyze a saved run.')
    ACTIVE_ROOT = Path(EXISTING_ROOT).expanduser().resolve()
    SCOPE = load_manifest(ACTIVE_ROOT / 'manifest.json')['scope']
else:
    fields = ('start_month', 'start_id', 'end_month', 'end_id')
    if all(selected_assignment.get(field) is not None for field in fields):
        SCOPE = _assignment_scope(selected_assignment, MONTH_END_IDS)
    elif ALLOW_NETWORK:
        raise ValueError('Fill all four range fields for SELECTED_MEMBER before collecting.')
    if RUN_MODE == 'cold':
        ACTIVE_ROOT = BASE_DATA_DIR / 'corpus' / f'member-{member_number}' / RUN_FOLDER
    else:
        if EXISTING_ROOT is None:
            raise ValueError('Set EXISTING_ROOT to resume a saved run.')
        ACTIVE_ROOT = Path(EXISTING_ROOT).expanduser().resolve()
        saved_scope = load_manifest(ACTIVE_ROOT / 'manifest.json')['scope']
        if AUTO_PLAN_SESSIONS:
            if not set(scope_ids(saved_scope)).issubset(scope_ids(ASSIGNMENT_SCOPE)):
                raise ValueError('Saved resume scope is outside this member assignment.')
        elif SCOPE is not None and not set(scope_ids(SCOPE)).issubset(scope_ids(saved_scope)):
            raise ValueError('Configured segment must belong to the saved resume scope.')
        # A prior cold session may also contain automatically carried retry IDs.
        SCOPE = saved_scope

display(pd.DataFrame(ASSIGNMENTS).style.format(na_rep='—'))
print('Member:', SELECTED_MEMBER, '| Mode:', RUN_MODE, '| Root:', ACTIVE_ROOT)
if SCOPE is None:
    print('Assignment not entered; collection is disabled until its four fields are filled.')
else:
    print('Scope:', SCOPE['mode'], '| IDs:', len(SCOPE['ids']) if SCOPE['mode'] == 'ids' else f"{SCOPE['start_id']} → {SCOPE['end_id']}")

### Google Drive

Kết nối và kiểm tra quyền ghi Drive trước khi thu thập. Cell dừng nếu không xác nhận được quyền truy cập.


In [ ]:
# Mount and verify Drive write access before collection.
DRIVE_READY = False
if SAVE_TO_DRIVE and IN_COLAB and ALLOW_NETWORK:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    my_drive = (DRIVE_MOUNT_POINT / 'MyDrive').resolve()
    if not os.path.ismount(DRIVE_MOUNT_POINT) or not my_drive.is_dir():
        raise OSError('Drive mount was not confirmed; collection has not started.')
    if not DRIVE_BACKUP_DIR.resolve().is_relative_to(my_drive):
        raise ValueError('DRIVE_BACKUP_DIR must be inside mounted MyDrive.')
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    probe = DRIVE_BACKUP_DIR / ('.write-check-' + uuid.uuid4().hex)
    probe.write_text('arxiv-drive-ready', encoding='utf-8')
    if probe.read_text(encoding='utf-8') != 'arxiv-drive-ready':
        raise OSError('Drive write/read check failed; collection has not started.')
    probe.unlink()
    DRIVE_READY = True
    print('Drive ready before collection:', DRIVE_BACKUP_DIR)
else:
    print('Drive not connected: local/offline mode or SAVE_TO_DRIVE=False.')


## 3. Benchmark environment

Benchmark chạy trên Colab CPU, đo từ chuẩn bị index đến hết discovery, crawl và checkpoint; không gồm cài đặt, biểu đồ và backup cuối.
RAM đo bằng process-tree RSS; đĩa gồm corpus, index và cache, lấy mẫu mỗi 0,25 giây. Cold benchmark dùng index mới và cache ngày v1 rỗng.


In [ ]:
def git_value(*arguments):
    try:
        result = subprocess.run(['git', '-C', str(PROJECT_DIR), *arguments],
                                capture_output=True, text=True, timeout=5, check=False)
        return result.stdout.strip() if result.returncode == 0 else None
    except (OSError, subprocess.TimeoutExpired):
        return None

cpu_only = _cpu_only_environment()
existing_disk_path = next(parent for parent in [BASE_DATA_DIR, *BASE_DATA_DIR.parents] if parent.exists())
versions = {}
for package in ('requests', 'psutil', 'pandas', 'matplotlib', 'ipykernel', 'jupyterlab'):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None

environment_before = {
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    'runtime': 'Colab' if IN_COLAB else 'local', 'platform': platform.platform(),
    'python': platform.python_version(), 'machine': platform.machine(),
    'processor': platform.processor() or None, 'cpu_logical_count': psutil.cpu_count(),
    'cpu_physical_count': psutil.cpu_count(logical=False),
    'machine_ram_bytes': psutil.virtual_memory().total,
    'filesystem_total_bytes': shutil.disk_usage(existing_disk_path).total,
    'filesystem_free_bytes_before': shutil.disk_usage(existing_disk_path).free,
    'cpu_only_detected': cpu_only, 'package_versions': versions,
    'git_commit': git_value('rev-parse', 'HEAD'),
    'git_worktree_dirty': None if git_value('status', '--porcelain') is None
                          else bool(git_value('status', '--porcelain')),
}
if REQUIRE_COLAB_CPU and RUN_MODE != 'analyze' and (not IN_COLAB or not cpu_only):
    raise RuntimeError('Final benchmark requires a Colab CPU-only runtime. Select accelerator None.')
display(pd.DataFrame([{'field': key, 'value': str(value)} for key, value in environment_before.items()
                      if key != 'package_versions']))
display(pd.DataFrame(list(versions.items()), columns=['package', 'installed_version']))

## 4. Collection

OAI-PMH `arXivRaw` cung cấp metadata và lịch sử phiên bản; source được tải theo từng version, references lấy qua Semantic Scholar.
Ngày v1 của references lấy từ cache, Cornell Kaggle index hoặc OAI. Chỉ giữ `.tex`/`.bib`, nguyên nội dung.
Các request arXiv dùng một kết nối, cách nhau ít nhất 3,1 giây; giới hạn áp dụng chung cho các máy của nhóm.


### Session planning and retry

Planner chọn bài mới và tối đa `MAX_RETRY_PAPERS` bài pending; `RETRY_ONLY=True` chỉ chạy pending.
Tác vụ 404/410 đã xác định được bỏ qua; 429, timeout và 5xx được retry. Bài chỉ cách ly khi mọi tác vụ còn thiếu đều là lỗi vĩnh viễn; `no_tex` giữ record rỗng.


In [ ]:
# Session timer includes planning and retry restoration.
SESSION_STARTED_AT = time.perf_counter() if RUN_MODE != 'analyze' else None
session_plan = None
SESSION_STATE_PATH = None
SESSION_COMPLETE = False
RETRY_SEEDED = False
if RUN_MODE != 'analyze' and ALLOW_NETWORK and AUTO_PLAN_SESSIONS and not DRIVE_READY:
    raise OSError('Automatic planning requires Drive progress; connect Drive before collection.')
if RUN_MODE != 'analyze' and ALLOW_NETWORK and IN_COLAB and SAVE_TO_DRIVE and (AUTO_RETRY_FROM_DRIVE or AUTO_PLAN_SESSIONS):
    if not DRIVE_READY:
        raise OSError('Connect Drive before registering work.')
    if RUN_MODE == 'cold' and ACTIVE_ROOT.exists() and any(ACTIVE_ROOT.iterdir()):
        raise ValueError('Cold session needs a new root; rerun configuration only to start a NEW session.')
    SESSION_STATE_PATH = DRIVE_BACKUP_DIR / 'retry-state' / f'member-{member_number}.json'
    migrated = migrate_permanent_failures(SESSION_STATE_PATH, member=SELECTED_MEMBER)
    if migrated:
        print('Đã cách ly từ backup cũ:', migrated, 'bài; không retry tự động.')
    overview = progress(SESSION_STATE_PATH, member=SELECTED_MEMBER, assignment_scope=ASSIGNMENT_SCOPE,
                        previous_assignment_scope=PREVIOUS_ASSIGNMENT_SCOPE)
    display(pd.DataFrame([{k:v for k,v in overview.items() if k not in ('pending_ids','unseen_ids')}]))
    print(f"Saved complete: {overview['completed']}/{overview['total']}; pending: {overview['pending']}; cách ly: {overview['quarantined']}; unseen: {overview['unseen']}")
    if RUN_MODE == 'cold' and AUTO_PLAN_SESSIONS:
        session_plan = plan_session(SESSION_STATE_PATH, member=SELECTED_MEMBER, assignment_scope=ASSIGNMENT_SCOPE,
                                    previous_assignment_scope=PREVIOUS_ASSIGNMENT_SCOPE,
                                    new_limit=NEW_PAPERS_PER_SESSION, retry_limit=(MAX_RETRY_PAPERS if AUTO_RETRY_FROM_DRIVE else 0), retry_only=RETRY_ONLY)
    else:
        session_plan = prepare_session(SESSION_STATE_PATH, member=SELECTED_MEMBER, scope=SCOPE, extend=(RUN_MODE == 'cold'))
    SCOPE = session_plan['scope']
    SESSION_COMPLETE = SCOPE is None
    if SESSION_COMPLETE:
        print('No work selected: check completed/pending/unseen and retry settings. No crawl will start.')
    else:
        restored_files = 0
        if RUN_MODE == 'cold':
            restored_files = seed_pending(session_plan, ACTIVE_ROOT)
            RETRY_SEEDED = ACTIVE_ROOT.exists() and any(ACTIVE_ROOT.iterdir())
            if RETRY_SEEDED:
                save_manifest(initialize_manifest(SCOPE, run_id=str(uuid.uuid4())), ACTIVE_ROOT / 'manifest.json')
        session_plan['restored_files'] = restored_files
        print('This session: new =',len(session_plan['new_ids']),'retry =',len(session_plan['retry_ids']),
              'actual unique IDs =',len(session_plan['requested_ids']))
        print('First/last selected ID:',session_plan['requested_ids'][0],session_plan['requested_ids'][-1])
        print('Restored files:',restored_files,'| Drive state:',SESSION_STATE_PATH)
        if session_plan['deferred_ids']:
            print('Pending IDs deferred by retry limit:',len(session_plan['deferred_ids']))
else:
    print('Drive progress/planning skipped: analyze/offline or automatic retry disabled.')


In [ ]:
run_result = None
manifest = None
report_dir = None
scope_purpose_for_report = SCOPE_PURPOSE

if RUN_MODE == 'analyze':
    manifest = load_manifest(ACTIVE_ROOT / 'manifest.json')
    run_result = json.loads((ACTIVE_ROOT / 'metrics/run_summary.json').read_text(encoding='utf-8'))
    if run_result['run_id'] != manifest['run_id']:
        raise ValueError('Stored summary belongs to an older run. Resume the interrupted pipeline before reporting.')
    print('Loaded saved run:', run_result['run_id'])
elif SESSION_COMPLETE:
    print('No selected work; existing Drive results are preserved.')
elif ALLOW_NETWORK:
    if SCOPE is None:
        raise ValueError('Fill the selected assignment before collecting.')
    print('Starting collection; data root:', ACTIVE_ROOT, flush=True)
    background = (BackgroundCheckpoints(ACTIVE_ROOT,session_plan,POLICY,every=CHECKPOINT_EVERY_PAPERS)
                  if BACKGROUND_CHECKPOINTS and session_plan and session_plan.get('ticket') and DRIVE_READY else None)
    date_config = None
    if USE_HYBRID_REFERENCE_DATES:
        date_store = DRIVE_BACKUP_DIR if DRIVE_READY else BASE_DATA_DIR
        date_config = dict(
            use_kaggle=USE_KAGGLE_REFERENCE_DATES,
            index_store=date_store / 'metadata-index',
            local_dir=BASE_DATA_DIR / 'metadata-work' / RUN_FOLDER,
            cache_dir=Path(V1_CACHE_DIR) if V1_CACHE_DIR is not None else date_store / 'v1-cache' / f'member-{member_number}',
            snapshot=KAGGLE_SNAPSHOT_PATH, auto_download=AUTO_DOWNLOAD_KAGGLE,
            refresh=REFRESH_KAGGLE_INDEX, max_age_days=V1_CACHE_MAX_AGE_DAYS,
        )
    run_result = run_pipeline(deepcopy(SCOPE), ACTIVE_ROOT, policy=POLICY,
                              reference_dates_config=date_config,
                              resume=(RUN_MODE == 'resume' or RETRY_SEEDED), checkpoint=background,
                              retry_permanent=RETRY_PERMANENT_FAILURES)
    manifest = load_manifest(ACTIVE_ROOT / 'manifest.json')
    assert run_result['run_id'] == manifest['run_id']
    print('Completed:', run_result['status'], '| run_id:', run_result['run_id'])
else:
    print('Network disabled. Enable ALLOW_NETWORK for collection, or use analyze with an existing root.')

if run_result is not None:
    assert summarize_corpus(manifest) == run_result['summary']
    report_dir = BASE_DATA_DIR / 'reports' / run_result['run_id']
    report_dir.mkdir(parents=True, exist_ok=True)
    export_report_data(manifest, run_result['benchmark'], report_dir)
    for filename, value in (('run_summary.json', run_result), ('manifest.json', manifest)):
        (report_dir / filename).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    if RUN_MODE != 'analyze':
        (report_dir / 'environment.json').write_text(json.dumps(environment_before, indent=2), encoding='utf-8')
        (report_dir / 'configuration.json').write_text(json.dumps({
            'scope': manifest['scope'], 'scope_purpose': SCOPE_PURPOSE,
            'metadata_source': METADATA_SOURCE, 'oai_endpoint': OAI_ENDPOINT, 'metadata_prefix': METADATA_PREFIX,
            'requested_mode': RUN_MODE, 'policy': POLICY, 'data_root': str(ACTIVE_ROOT),
            'require_colab_cpu': REQUIRE_COLAB_CPU,
            'save_to_drive': SAVE_TO_DRIVE, 'drive_backup_dir': str(DRIVE_BACKUP_DIR),
            'auto_retry_from_drive': AUTO_RETRY_FROM_DRIVE, 'session_plan': session_plan,
            'auto_plan': AUTO_PLAN_SESSIONS, 'new_limit': NEW_PAPERS_PER_SESSION,
            'retry_only': RETRY_ONLY, 'retry_limit': MAX_RETRY_PAPERS,
            'retry_permanent_failures': RETRY_PERMANENT_FAILURES,
            'background_checkpoints': BACKGROUND_CHECKPOINTS, 'checkpoint_every_papers': CHECKPOINT_EVERY_PAPERS,
            'selected_assignment': selected_assignment, 'assignments': ASSIGNMENTS,
            'month_end_ids': MONTH_END_IDS,
            'reference_dates_config': {k: str(v) if isinstance(v, Path) else v for k,v in (date_config or {}).items()},
        }, indent=2), encoding='utf-8')
    else:
        (report_dir / 'analysis_environment.json').write_text(json.dumps(environment_before, indent=2), encoding='utf-8')
        stored_configuration = report_dir / 'configuration.json'
        scope_purpose_for_report = (json.loads(stored_configuration.read_text())['scope_purpose']
                                    if stored_configuration.is_file() else 'unrecorded')
    for filename in ('resource_samples.json', 'resource_errors.json', 'reference_date_stats.json'):
        source = ACTIVE_ROOT / 'metrics' / filename
        if source.is_file():
            shutil.copyfile(source, report_dir / filename)
    (report_dir / 'metadata_stats.json').write_text(json.dumps(run_result['metadata_stats'], indent=2), encoding='utf-8')
    if session_plan is not None:
        counts = write_session_breakdown(report_dir,session_plan,manifest)
        display(pd.DataFrame([counts]))
    print('Per-run report snapshot:', report_dir)
if run_result is not None:
    show_run_dashboard()


## 5. Coverage and quality

Tỷ lệ nguồn tính theo số version kỳ vọng; tỷ lệ bài và references tính theo số bài discovered. `no_tex` được tính vào độ phủ xử lý nguồn, không tính là có TeX.
Kích thước trước/sau dùng cùng tập đo. Giữ mọi reference ở local, đồng thời tách pool có arXiv ID; bảng dưới hiển thị tối đa 5 dòng mẫu.


In [ ]:
papers = pd.DataFrame()
versions_table = pd.DataFrame()
references_table = pd.DataFrame()
all_references_table = pd.DataFrame()
resource_samples = []
resource_errors = []

if run_result is not None:
    summary = run_result['summary']
    discovery = manifest['discovery']
    display(pd.DataFrame([run_result['metadata_stats']]))
    coverage = [
        ('Requested IDs', discovery['requested_count'] if discovery else None),
        ('Discovered papers', summary['papers_discovered']),
        ('Discovery issues', len(discovery['issues']) if discovery else None),
        ('Full TeX papers / discovered', f"{summary['full_source_papers']}/{summary['papers_discovered']}"),
        ('Source versions: success / no_tex / failed / pending',
         ' / '.join(str(summary['versions_' + s]) for s in ('success', 'no_tex', 'failed', 'pending'))),
        ('References: success / partial / failed / not_found / pending',
         ' / '.join(str(summary['reference_jobs_' + s]) for s in ('success', 'partial', 'failed', 'not_found', 'pending'))),
        ('Source coverage (includes verified no_tex)', f"{summary['source_coverage_rate']:.1%}" if summary['source_coverage_rate'] is not None else '—'),
        ('Mean paired source: before → after', format_bytes(summary['average_source_bytes_before']) + ' → ' + format_bytes(summary['average_source_bytes_after'])),
        ('Mean paper processing', format_duration(summary['average_paper_processing_seconds'])),
    ]
    display(pd.DataFrame(coverage, columns=['metric', 'value']))
    for name, label in [('papers', 'Bài'), ('versions', 'Phiên bản'),
                        ('references', 'Reference arXiv'), ('references_all', 'Tất cả reference local')]:
        table = pd.read_csv(report_dir / f'{name}.csv')
        if name == 'papers': papers = table
        elif name == 'versions': versions_table = table
        elif name == 'references': references_table = table
        else: all_references_table = table
        display(Markdown(f'**{label}:** {len(table):,} dòng; mẫu tối đa 5 dòng. Đầy đủ: `{name}.csv`.'))
        display(preview_style(table))
    print('Toàn bộ chỉ số và mẫu số: summary.json. Giá trị gốc trong CSV/JSON giữ nguyên đơn vị byte/giây.')
else:
    print('Coverage tables appear after a collected or analyzed run.')


## 6. Integrity audit

Kiểm tra metadata, ID, SHA-256 và kích thước file; hiển thị 5 dòng mẫu cùng toàn bộ nhóm lỗi.
Tỷ lệ audit tính trên hạng mục đã kiểm, không đại diện độ phủ corpus. Kết quả đầy đủ nằm trong CSV xuất kèm.


In [ ]:
audit_rows = []
issue_rows = []
if run_result is not None:
    for issue in (manifest['discovery'] or {}).get('issues', []):
        issue_rows.append(dict(arxiv_id=issue['arxiv_id'], task='discovery',
                               status=issue['status'], error=issue['error']))
    for identifier, state in manifest['papers'].items():
        directory = paper_directory(identifier, ACTIVE_ROOT)
        if any(state['downloads'].values()) or state['references'] is not None:
            try:
                valid = json.loads((directory / 'metadata.json').read_text(encoding='utf-8')) == state['record']['metadata']
                audit_rows.append(dict(arxiv_id=identifier, artifact='metadata', valid=valid, detail='JSON matches record'))
            except (OSError, ValueError) as exc:
                audit_rows.append(dict(arxiv_id=identifier, artifact='metadata', valid=False, detail=type(exc).__name__))
        for version, result in state['downloads'].items():
            if result is None or result['status'] == 'failed':
                issue_rows.append(dict(arxiv_id=identifier, task=f'source v{version}',
                                       status='pending' if result is None else 'failed',
                                       error=None if result is None else result['error']))
                continue
            try:
                receipt = json.loads((directory / 'sources' / f'.v{version}.receipt.json').read_text(encoding='utf-8'))
                target = directory / 'sources' / f'v{version}'
                retained = [p for p in target.rglob('*') if p.is_file()]
                inventory = {p.relative_to(target).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in retained}
                cached_source = cached_source_result(identifier, int(version), target)
                valid = (cached_source is not None and cached_source['status'] == result['status']
                         and target.is_dir() and receipt['schema_version'] == '3.0'
                         and receipt['arxiv_id'] == identifier and receipt['version'] == int(version)
                         and all(p.suffix.lower() in {'.tex', '.bib'} for p in retained)
                         and inventory == receipt['files_sha256'] and result['file_count'] == len(retained)
                         and result['source_bytes_after'] == sum(p.stat().st_size for p in retained)
                         and (result['status'] != 'no_tex' or not retained))
                audit_rows.append(dict(arxiv_id=identifier, artifact=f'source v{version}', valid=valid,
                                       detail=f'{len(retained)} retained files; inventory/hash/bytes'))
            except (OSError, ValueError, KeyError, TypeError) as exc:
                audit_rows.append(dict(arxiv_id=identifier, artifact=f'source v{version}', valid=False,
                                       detail=type(exc).__name__))
        refs = state['references']
        if refs is None or refs['status'] != 'success':
            issue_rows.append(dict(arxiv_id=identifier, task='references',
                                   status='pending' if refs is None else refs['status'],
                                   error=None if refs is None else refs['error']))
        else:
            try:
                cached_refs = cached_reference_result(identifier, directory, require_fresh=False)
                valid = (cached_refs is not None and cached_refs['references'] == refs['references']
                         and cached_refs['all_references'] == refs['all_references']
                         and cached_refs['metadata_source'] == refs['metadata_source'])
                audit_rows.append(dict(arxiv_id=identifier, artifact='references', valid=valid,
                                       detail=f"{refs['kept_references']} unique arXiv IDs; both stores checked"))
            except (OSError, ValueError, KeyError, TypeError) as exc:
                audit_rows.append(dict(arxiv_id=identifier, artifact='references', valid=False, detail=type(exc).__name__))
        try:
            checker = OaiClient(policy=POLICY, cache_dir=ACTIVE_ROOT / 'metadata_cache')
            cached_record = checker._cached(identifier, require_fresh=False)
            valid = cached_record == state['record']
            checked = 1
            if refs and refs['status'] == 'success' and refs['metadata_source'] == METADATA_SOURCE:
                for ref_key, ref_data in refs['references'].items():
                    ref_id = ref_key if '/' in ref_key else ref_key.replace('-', '.')
                    cached_ref = checker._cached(ref_id, require_fresh=False)
                    valid = valid and cached_ref is not None and cached_ref['metadata']['submission_date'] == ref_data['submission_date']
                    checked += 1
            audit_rows.append(dict(arxiv_id=identifier, artifact='OAI metadata checkpoints', valid=bool(valid),
                                   detail=f'{checked} OAI records; hybrid reference evidence checked in receipt'))
        except (OSError, ValueError, KeyError, TypeError) as exc:
            audit_rows.append(dict(arxiv_id=identifier, artifact='OAI metadata checkpoints', valid=False, detail=type(exc).__name__))
    audit_table = pd.DataFrame(audit_rows, columns=['arxiv_id', 'artifact', 'valid', 'detail'])
    # One shared classification drives notebook, queue and CSV exports.
    issue_rows = failure_rows(manifest)
    issue_rows.extend(dict(arxiv_id=r['arxiv_id'], task=r['artifact'], status='integrity_failed',
                           category='review', code='integrity', reason='Integrity audit không hợp lệ', error=r['detail'])
                      for r in audit_rows if not r['valid'])
    issues_table = pd.DataFrame(issue_rows, columns=['arxiv_id', 'task', 'status', 'category', 'code', 'reason', 'error'])
    display(preview_style(audit_table))
    checked = len(audit_table)
    valid_count = int(audit_table['valid'].sum()) if checked else 0
    audit_pct = f'{valid_count / checked:.1%}' if checked else 'chưa có dữ liệu'
    print(f'Kiểm toán {valid_count}/{checked} hạng mục hợp lệ ({audit_pct}); chi tiết: integrity_audit.csv.')
    grouped_issues = export_issue_groups(issues_table, report_dir / 'issue_groups.csv')
    if not grouped_issues.empty:
        display(preview_style(grouped_issues, limit=len(grouped_issues)))
        print(f'Hiển thị đầy đủ {len(grouped_issues)}/{len(grouped_issues)} nhóm lỗi; chi tiết: issue_groups.csv.')
        print('Đếm bài duy nhất trong từng nhóm; các nhóm có thể trùng bài.')
    else:
        print('Không có tác vụ tồn đọng hoặc no_tex trong phiên này.')
    audit_table.to_csv(report_dir / 'integrity_audit.csv', index=False)
    issues_table[issues_table['status'] != 'no_tex'].to_csv(report_dir / 'outstanding_tasks.csv', index=False)
    issues_table[issues_table['category'] == 'permanent'].to_csv(report_dir / 'permanent_failures.csv', index=False)
    print('Chi tiết lỗi: outstanding_tasks.csv; quyết định không retry: permanent_failures.csv (gồm no_tex).')
else:
    print('Integrity audit awaits real outputs.')

## 7. Corpus figures

Phân bố kết quả nguồn, references, kích thước và thời gian xử lý; xuất PNG/SVG.


In [ ]:
figure_paths = []
def save_figure(figure, name):
    target = report_dir / 'figures'
    target.mkdir(exist_ok=True)
    for extension in ('png', 'svg'):
        path = target / f'{name}.{extension}'
        figure.savefig(path, bbox_inches='tight')
        figure_paths.append(path)
    plt.show()
    plt.close(figure)

if run_result is not None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout='constrained')
    source_statuses = ['success', 'no_tex', 'failed', 'pending']
    colors = ['#207561', '#d69c2f', '#c54c4c', '#8392a5']
    axes[0, 0].bar(source_statuses, [summary['versions_' + s] for s in source_statuses], color=colors)
    axes[0, 0].set(title='Source versions by outcome', ylabel='Versions')
    reference_statuses = ['success', 'partial', 'failed', 'not_found', 'pending']
    axes[0, 1].bar(reference_statuses, [summary['reference_jobs_' + s] for s in reference_statuses],
                   color=['#207561', '#d69c2f', '#c54c4c', '#7867a0', '#8392a5'])
    axes[0, 1].tick_params(axis='x', labelrotation=20)
    axes[0, 1].set(title='Reference jobs by outcome', ylabel='Papers')
    paired = versions_table.dropna(subset=['source_bytes_before', 'source_bytes_after'])
    if not paired.empty:
        values = [paired['source_bytes_before'].mean() / 1024, paired['source_bytes_after'].mean() / 1024]
        axes[1, 0].bar(['Before filtering', 'After filtering'], values, color=['#4979a7', '#207561'])
        axes[1, 0].set(title=f'Mean paired source size (n={len(paired)} versions)', ylabel='KiB / version')
    else:
        axes[1, 0].text(.5, .5, 'No paired size measurements', ha='center', va='center', transform=axes[1, 0].transAxes)
        axes[1, 0].set_title('Source size: unavailable')
        axes[1, 0].set_axis_off()
    durations = papers['elapsed_seconds'].dropna()
    if not durations.empty:
        axes[1, 1].hist(durations, bins=min(15, max(1, len(durations))), color='#4979a7', edgecolor='white')
        axes[1, 1].set(title=f'Current-run paper processing time (n={len(durations)})', xlabel='Seconds', ylabel='Papers')
    else:
        axes[1, 1].text(.5, .5, 'No completed paper timings', ha='center', va='center', transform=axes[1, 1].transAxes)
        axes[1, 1].set_title('Paper processing time: unavailable')
        axes[1, 1].set_axis_off()
    for axis in (axes[0, 0], axes[0, 1], axes[1, 1]):
        axis.yaxis.set_major_locator(MaxNLocator(integer=True))
    for axis in axes.flat:
        axis.grid(axis='y', alpha=.2)
        axis.set_axisbelow(True)
    fig.suptitle(f"Corpus outcomes — {run_result['benchmark']['cache_mode']} / {scope_purpose_for_report}")
    save_figure(fig, 'corpus_outcomes')
else:
    print('Figures await collected data; no numbers are generated.')

## 8. End-to-end benchmark

Báo cáo thời gian discovery/toàn pipeline/trung bình mỗi bài, RAM và dung lượng đĩa. Cold và resume được ghi riêng.


In [ ]:
benchmark_checks = {}
if run_result is not None:
    benchmark = run_result['benchmark']
    samples_path = ACTIVE_ROOT / 'metrics/resource_samples.json'
    errors_path = ACTIVE_ROOT / 'metrics/resource_errors.json'
    resource_samples = json.loads(samples_path.read_text()) if samples_path.is_file() else []
    resource_errors = json.loads(errors_path.read_text()) if errors_path.is_file() else []
    date_stats_path = ACTIVE_ROOT / 'metrics/reference_date_stats.json'
    reference_date_stats = json.loads(date_stats_path.read_text()) if date_stats_path.is_file() else {}
    if reference_date_stats:
        display(pd.DataFrame([reference_date_stats.get('lookups', {})]))
        print('Kaggle preparation:', format_duration(reference_date_stats.get('elapsed_seconds')),
              '| reused index:', reference_date_stats.get('reused_index'))
    unit_rows = [
        ('End-to-end wall', benchmark['wall_seconds'], 's'),
        ('Discovery', (manifest['discovery'] or {}).get('elapsed_seconds'), 's'),
        ('Mean paper processing', summary['average_paper_processing_seconds'], 's'),
        ('Observed RAM peak', benchmark['ram_peak_bytes'], 'byte'),
        ('Sample RAM mean', benchmark['ram_mean_bytes'], 'byte'),
        ('Observed disk peak', benchmark['disk_peak_bytes'], 'byte'),
        ('Final disk', benchmark['final_disk_bytes'], 'byte'),
        ('Sampling interval', benchmark['sampling_interval_seconds'], 's'),
        ('Recorded samples', len(resource_samples) if samples_path.is_file() else None, 'samples'),
    ]
    formatted_rows = [(label, format_bytes(value) if unit == 'byte' else
                       format_duration(value) if unit == 's' else (str(value) if value is not None else '—'))
                      for label, value, unit in unit_rows]
    display(pd.DataFrame(formatted_rows, columns=['measurement', 'value']))
    display(pd.DataFrame([{'field': key, 'value': benchmark[key]} for key in ('environment', 'cpu_only', 'cache_mode')]))
    if resource_errors:
        display(preview_table(pd.DataFrame({'sampling_error': resource_errors})))
        print('Sampling errors:', len(resource_errors), '| full log: resource_errors.json')
    elif not errors_path.is_file():
        print('Sampling error log is unavailable for this historical run.')
    else:
        print('Sampling error log: no errors.')
    session_metadata = report_dir / 'session_plan.json'
    report_session_plan = json.loads(session_metadata.read_text(encoding='utf-8')) if session_metadata.is_file() else {}
    cache_rows = [value for state in manifest['papers'].values()
                  for value in [*state['downloads'].values(), state['references']] if value is not None]
    display(pd.DataFrame([{'reported_tasks': len(cache_rows),
                           'cache_hits': sum(value['used_cache'] for value in cache_rows),
                           'reported_attempts': sum(value['attempts'] for value in cache_rows)}]))
    benchmark_checks = {
        'Assigned segment recorded': scope_purpose_for_report == 'assigned',
        'Colab runtime recorded': 'Colab' in benchmark['environment'],
        'CPU-only runtime': benchmark['cpu_only'],
        'Cold-cache measurement': benchmark['cache_mode'] == 'cold',
        'Reference-date setup measured': reference_date_stats.get('preparation_included_in_benchmark', True),
        'No carried retry IDs': not report_session_plan.get('retry_ids'),
        'Operational success': run_result['status'] == 'success',
        'Scope complete without issues': bool(manifest['discovery'] and manifest['discovery']['scope_complete']
                                               and not manifest['discovery']['issues']),
        'Resource samples recorded': bool(resource_samples),
        'Resource errors absent': errors_path.is_file() and not resource_errors,
        'Current outputs pass integrity audit': bool(audit_rows) and all(row['valid'] for row in audit_rows),
    }
    display(pd.DataFrame(list(benchmark_checks.items()), columns=['final_benchmark_condition', 'satisfied']))
    print('Eligible as Colab benchmark for this assigned segment:', all(benchmark_checks.values()))
    (report_dir / 'benchmark_checks.json').write_text(json.dumps(benchmark_checks, indent=2))
else:
    print('Benchmark table awaits a measured run; setup does not measure a corpus benchmark.')

In [ ]:
if run_result is not None and resource_samples:
    samples_table = pd.DataFrame(resource_samples).sort_values('elapsed_seconds')
    samples_table.to_csv(report_dir / 'resource_samples.csv', index=False)
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, layout='constrained')
    axes[0].plot(samples_table['elapsed_seconds'], samples_table['ram_bytes'] / 1024**2, color='#4979a7', linewidth=1.5)
    axes[0].set(title='Process-tree RSS during collection', ylabel='MiB')
    axes[1].plot(samples_table['elapsed_seconds'], samples_table['disk_bytes'] / 1024**2, color='#207561', linewidth=1.5)
    axes[1].set(title='Files under the run root, including staging', ylabel='MiB', xlabel='Elapsed seconds')
    for axis in axes:
        axis.grid(alpha=.2)
    fig.suptitle(f"Observed resources — {benchmark['environment']} / {benchmark['cache_mode']}", fontsize=10)
    save_figure(fig, 'resource_traces')
elif run_result is not None:
    print('No stored resource samples: trace plot omitted. Stored benchmark fields remain unchanged.')

## 9. Report export

Xuất bảng, số đo, audit và hình theo `run_id`. `report.zip` chứa báo cáo; source corpus được lưu riêng.


In [ ]:
report_zip = None
if run_result is not None:
    if session_plan is not None:
        invalid_ids = {row['arxiv_id'] for row in audit_rows if not row['valid']}
        write_session_breakdown(report_dir,session_plan,manifest,invalid_ids)
    scope_lines = f"{len(scope_ids(manifest['scope']))} requested IDs; exact scope in manifest.json"
    status_text = (
        f"# Run {run_result['run_id']}\n\n"
        f"Status: {run_result['status']}\n\nScope: {scope_lines}\n\n"
        f"Scope purpose: {scope_purpose_for_report}\n\n"
        'For new/retried IDs and their backup origins see session_plan.json when present.\n\n'
        f"Environment: {benchmark['environment']}\n\nCache mode: {benchmark['cache_mode']}\n\n"
        f"Pipeline wall: {format_duration(benchmark['wall_seconds'])} (min:sec)\n\n"
        f"Eligible as Colab benchmark for this assigned segment: {all(benchmark_checks.values())}\n\n"
        'See outstanding_tasks.csv, integrity_audit.csv and benchmark_checks.json for limitations.\n'
        'Source corpus is kept separately; this report bundle is not the corpus.\n'
    )
    (report_dir / 'README.md').write_text(status_text, encoding='utf-8')
    if CREATE_REPORT_ZIP:
        archive_dir = BASE_DATA_DIR / 'archives'
        archive_dir.mkdir(parents=True, exist_ok=True)
        report_zip = Path(shutil.make_archive(str(archive_dir / f"report-{run_result['run_id']}"), 'zip', root_dir=report_dir))
        print('Report ZIP:', report_zip, '| size:', format_bytes(report_zip.stat().st_size))

else:
    print('No run exported. Configure collection or analyze an existing measured run.')

### Drive backup

Checkpoint lưu sau mỗi `CHECKPOINT_EVERY_PAPERS` bài; cell dưới lưu thêm corpus và báo cáo cuối.
Backup hoàn tất khi hiển thị **VERIFIED**. File nguồn dùng cho M2 nằm trong corpus ZIP của checkpoint hoặc backup đầy đủ; Corpus ZIP tại `session-reports` chỉ chứa manifest.


In [ ]:
drive_backup = None
drive_backup_error = None
retry_queue_error = None
if run_result is None:
    print('Drive backup skipped: no collected/analyzed run.')
elif not SAVE_TO_DRIVE:
    print('Drive backup disabled. Keep a separate corpus backup for M2/M3.')
elif not IN_COLAB or not ALLOW_NETWORK:
    print('Drive backup skipped: requires Colab and ALLOW_NETWORK=True.')
else:
    try:
        if not DRIVE_READY or not os.path.ismount(DRIVE_MOUNT_POINT):
            raise OSError('Drive is no longer mounted; reconnect with the drive-connect cell.')
        print('Backing up corpus/report to Drive...', flush=True)
        drive_backup = backup_run(
            ACTIVE_ROOT, report_dir, DRIVE_BACKUP_DIR,
            run_id=run_result['run_id'], work_dir=BASE_DATA_DIR / 'backup-staging',
        )
        print('Drive backup VERIFIED:', drive_backup['directory'])
    except Exception as exc:
        # Avoid leaking authentication details from exception messages.
        drive_backup_error = type(exc).__name__
        print('DRIVE BACKUP FAILED:', drive_backup_error)
        print('Local corpus/report preserved:', ACTIVE_ROOT, report_dir)
        print('Pending IDs retained; resolve Drive error and rerun this cell.')
    if drive_backup is not None and session_plan is not None:
        try:
            audited_ids = {row['arxiv_id'] for row in audit_rows}
            invalid_ids = {row['arxiv_id'] for row in audit_rows if not row['valid']}
            invalid_ids.update(set(manifest['papers']) - audited_ids)
            retry_state = finish_session(SESSION_STATE_PATH, session_plan, manifest,
                                         drive_backup, invalid_ids=invalid_ids)
            print('Queue saved:', len(retry_state['pending_ids']), 'pending;',
                  len(retry_state['permanent_failures']), 'bài cách ly, không retry tự động.')
        except Exception as exc:
            retry_queue_error = type(exc).__name__
            print('RETRY QUEUE UPDATE FAILED:', retry_queue_error)
            print('Backup verified; queue update incomplete. Check Drive and rerun this cell.')

if run_result is not None:
    invalid_ids = {row['arxiv_id'] for row in audit_rows if not row['valid']}
    show_run_dashboard(drive_backup, invalid_ids)


In [ ]:
# Optional browser download of the report ZIP.
if DOWNLOAD_REPORT_ZIP and run_result is not None:
    if not IN_COLAB or report_zip is None:
        raise ValueError('Browser download requires Colab and CREATE_REPORT_ZIP=True.')
    from google.colab import files
    files.download(str(report_zip))


## 10. Review results

Đối chiếu số ID/version với phân công, kiểm tra `integrity_audit.csv` và các lỗi còn lại trong `outstanding_tasks.csv`.
Bài cách ly không tính là hoàn thành. `benchmark_checks.json` ghi điều kiện hợp lệ của benchmark Colab CPU.


## Documentation

[Hướng dẫn chạy và khôi phục](../../docs/milestone1/HUONG_DAN_SU_DUNG_NOTEBOOK.md) · [Thiết kế và nguồn tham khảo](../../docs/milestone1/KE_HOACH_MILESTONE_1.md) · [Reference dates](../../docs/milestone1/TOI_UU_REFERENCE_DATES.md).
